# 04 Moon Analysis

This notebook covers analysis techniques specific to lunar images surface detail, brightness profiles, edge detection, and crater analysis.

**Requirements:** astropy, numpy, matplotlib, scipy

In [ ]:
from astropy.io import fits
from astropy.visualization import LinearStretch, PercentileInterval
from scipy.ndimage import sobel
import numpy as np
import matplotlib.pyplot as plt

fits_file = "your_moon_image.fits"  # Change this to your file path

with fits.open(fits_file) as hdul:
    data   = hdul[0].data
    header = hdul[0].header

rgb = np.transpose(data, (1, 2, 0))

## Displaying the Moon

The Moon is very bright we use LinearStretch to preserve natural surface gradients without overexposing.

### Why LinearStretch?
Unlike Asinh (used for nebulae), LinearStretch keeps the brightness proportional better for studying surface detail.

In [ ]:
interval = PercentileInterval(99.0)  # Ignore top 1% of values to avoid overexposure
stretch  = LinearStretch()

rgb_moon = np.zeros_like(rgb)
for i in range(3):
    vmin, vmax      = interval.get_limits(rgb[:,:,i])
    normalized      = np.clip((rgb[:,:,i] - vmin) / (vmax - vmin), 0, 1)
    rgb_moon[:,:,i] = stretch(normalized)

plt.figure(figsize=(10, 10))
plt.imshow(rgb_moon, origin='lower')
plt.title('Moon Surface')
plt.axis('off')
plt.show()

## Brightness Profile

Plots brightness along a horizontal line through the center of the image.
Shows how brightness changes from one edge to the other useful for studying limb darkening.

In [ ]:
brightness_moon = np.mean(rgb_moon, axis=2)

mid_row = brightness_moon.shape[0] // 2  # Center row of the image
profile = brightness_moon[mid_row, :]

plt.figure(figsize=(12, 4))
plt.plot(profile, color='white', linewidth=1.5)
plt.fill_between(range(len(profile)), profile, alpha=0.3)
plt.title('Brightness Profile Horizontal Cross-Section')
plt.xlabel('X Position (pixels)')
plt.ylabel('Brightness')
plt.gca().set_facecolor('#111111')
plt.gcf().set_facecolor('#111111')
plt.tick_params(colors='white')
plt.show()

## Edge Detection

The Sobel filter detects sharp brightness changes highlighting crater rims and mountain edges on the lunar surface.

### How it works
 `sobel_x` detects horizontal edges
 `sobel_y` detects vertical edges
 `np.hypot` combines both into a single edge magnitude map

In [ ]:
sobel_x = sobel(brightness_moon, axis=1)  # Horizontal edges
sobel_y = sobel(brightness_moon, axis=0)  # Vertical edges
edges   = np.hypot(sobel_x, sobel_y)      # Combined edge magnitude

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(brightness_moon, cmap='gray', origin='lower')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(edges, cmap='hot', origin='lower')
axes[1].set_title('Edge Detection (Sobel)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
## Crater Region Analysis

Crop a specific region around a crater and analyze its brightness distribution.
High standard deviation = strong contrast = good surface detail.

Adjust `cx`, `cy`, and `radius` to match the crater location in your image.